# Training a neural network for text classification with PyTorch

This notebook classifies short English texts. The model reads
**"At the finish: Gniewomir, Onufry. Who was first? Onufry or Gniewomir?"** and picks the second option,
which is **Gniewomir**. If we swap the finishing order, the correct answer becomes Onufry,
even though the text still contains exactly the same tokens.

| | MNIST | Text classification |
|---|---|---|
| Input | 784 pixel values | 16 token IDs |
| Layers in `nn.Sequential` | Linear and ReLU | Token and position embeddings, self-attention, linear layer |
| Output | 10 logits for digits 0–9 | 2 logits for the first and second option |

Both examples use `DataLoader`, `CrossEntropyLoss`, the Adam optimizer and
the same training loop. The text dataset is generated inside the notebook. You can run it
top to bottom on a CPU or a GPU.

### 1. Load libraries and choose a device

In [12]:
from importlib.util import find_spec

if find_spec("torch") is None:
    %pip install -q torch

In [13]:
import re
from itertools import combinations

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

seed = 42
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(2)  # A small model only needs a few threads.

print("PyTorch", torch.__version__)
print("Device", device)

PyTorch 2.15.0+cu132
Device cuda


### 2. Prepare texts and labels

For every pair of names we create four texts, swapping the order both at the finish
and in the question. Label `0` means the first option and `1` means the second.
All examples built from the same pair go into the same split, so the test set contains
**new combinations of known names**.

In [15]:
names = (
    "Gniewomir Andrzej Gwidolf Onufry Wacław Warcisław Wiesław Bogumił "
    "Bolesław Bronisław Dobromir Kazimierz Lechosław Ludomir Mścisław Przemysław "
    "Radosław Siemowit Sobiesław Świętopełk Teodor Zdzisław Ziemowit Florian"
).split()


def make_text(first, second, option_a, option_b):
    return f"At the finish: {first}, {second}. Who was first? {option_a} or {option_b}?"


def make_examples(name_pairs):
    examples = []
    for a, b in name_pairs:
        examples.extend([
            (make_text(a, b, a, b), 0),
            (make_text(a, b, b, a), 1),
            (make_text(b, a, a, b), 1),
            (make_text(b, a, b, a), 0),
        ])
    return examples


pairs = list(combinations(names, 2))
split_rng = torch.Generator().manual_seed(seed)
indices = torch.randperm(len(pairs), generator=split_rng).tolist()
shuffled_pairs = [pairs[index] for index in indices]
split = int(0.8 * len(shuffled_pairs))

train_pairs = shuffled_pairs[:split]
test_pairs = shuffled_pairs[split:]
train_examples = make_examples(train_pairs)
test_examples = make_examples(test_pairs)

print(f"Train: {len(train_pairs)} pairs, {len(train_examples)} texts")
print(f"Test: {len(test_pairs)} pairs, {len(test_examples)} texts")
for text, label in train_examples[:2]:
    print(f"Label {label} → {text}")

Train: 220 pairs, 880 texts
Test: 56 pairs, 224 texts
Label 0 → At the finish: Przemysław, Ziemowit. Who was first? Przemysław or Ziemowit?
Label 1 → At the finish: Przemysław, Ziemowit. Who was first? Ziemowit or Przemysław?


#### From text to tensor

We split the text into words and punctuation marks. Every token from the training set
gets an ID. All texts come from one template and have the same length, so we can
stack them directly into a tensor of shape `[num_examples, num_tokens]`. In this example
we use the names from the `names` list.

In [17]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.casefold())

vocabulary = sorted({token for text, _ in train_examples for token in tokenize(text)})
token_to_id = {token: index for index, token in enumerate(vocabulary)}

def encode(text):
    return [token_to_id[token] for token in tokenize(text)]

def make_dataset(examples):
    token_ids = torch.tensor([encode(text) for text, _ in examples], dtype=torch.long)
    labels = torch.tensor([label for _, label in examples], dtype=torch.long)
    return TensorDataset(token_ids, labels)


train_dataset = make_dataset(train_examples)
test_dataset = make_dataset(test_examples)

example_text, example_label = train_examples[0]
example_ids, _ = train_dataset[0]
sequence_length = example_ids.numel()

print("Text      ", example_text)
print("Tokens    ", tokenize(example_text))
print("Token IDs ", example_ids.tolist())
print("Label     ", example_label)
print("Vocabulary size", len(vocabulary))

Text       At the finish: Przemysław, Ziemowit. Who was first? Przemysław or Ziemowit?
Tokens     ['at', 'the', 'finish', ':', 'przemysław', ',', 'ziemowit', '.', 'who', 'was', 'first', '?', 'przemysław', 'or', 'ziemowit', '?']
Token IDs  [5, 26, 10, 2, 21, 0, 33, 1, 30, 29, 11, 3, 21, 20, 33, 3]
Label      0
Vocabulary size 35


### 3. Build the network with `nn.Sequential`

1. **Token embeddings** turn every ID into a learned vector of length 48.
2. **Position embeddings** add a learned vector for each token's position.
   This gives the network information about token order.
3. **Self-attention** updates the representation of every token using
   all the other tokens. The layer has four attention heads and sees the whole text.
4. **Token selection** takes the updated vector of the first answer option.
   It always sits at the same place in the template, which ends with `gniewomir or onufry ?`.
5. **A linear layer** computes two logits from that vector.

In [18]:
class AddPositionEmbeddings(nn.Module):
    def __init__(self, max_length, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(max_length, embedding_dim)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.embedding(positions)


class SelfAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        # Queries (Q), keys (K) and values (V) all come from the same sequence.
        attended, _ = self.attention(x, x, x, need_weights=False)
        return self.norm(x + attended)  # Residual connection and normalization.


class SelectFirstOption(nn.Module):
    def forward(self, x):
        # The last four tokens are e.g. "gniewomir", "or", "onufry", "?".
        return x[:, -4, :]


embedding_dim = 48

model = nn.Sequential(
    nn.Embedding(len(vocabulary), embedding_dim),
    AddPositionEmbeddings(sequence_length, embedding_dim),
    SelfAttention(embedding_dim, num_heads=4),
    SelectFirstOption(),
    nn.Linear(embedding_dim, 2),
).to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(), lr=0.003)

In [6]:
# Show the tensor shape after every layer for one example.
x = example_ids.unsqueeze(0).to(device)
print(f"{'Token IDs':<25} {tuple(x.shape)}")
with torch.no_grad():
    for layer in model:
        x = layer(x)
        print(f"{type(layer).__name__:<25} {tuple(x.shape)}")

Token IDs                 (1, 16)
Embedding                 (1, 16, 48)
AddPositionEmbeddings     (1, 16, 48)
SelfAttention             (1, 16, 48)
SelectFirstOption         (1, 48)
Linear                    (1, 2)


### 4. Train the network

The MNIST recipe works for text too. For every batch we compute the logits and the loss,
reset the gradients, call `backward()` and update the parameters with `step()`.
`CrossEntropyLoss` takes the logits directly.

In [19]:
def forward(input_tokens):
    return model(input_tokens)

def train_on_batch(batch_tokens, batch_labels):
    logits = forward(batch_tokens)
    loss = loss_function(logits, batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [20]:
batch_size = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_epochs = 12

for epoch_index in range(num_epochs):
    model.train()
    total_epoch_loss = 0.0

    for batch_tokens, batch_labels in train_loader:
        loss = train_on_batch(batch_tokens.to(device), batch_labels.to(device))
        total_epoch_loss += loss * len(batch_labels)

    average_epoch_loss = total_epoch_loss / len(train_dataset)
    print(f"Epoch {epoch_index + 1:2d}/{num_epochs} | average loss {average_epoch_loss:.4f}")

Epoch  1/12 | average loss 0.7435
Epoch  2/12 | average loss 0.6446
Epoch  3/12 | average loss 0.3008
Epoch  4/12 | average loss 0.0476
Epoch  5/12 | average loss 0.0327
Epoch  6/12 | average loss 0.0218
Epoch  7/12 | average loss 0.0090
Epoch  8/12 | average loss 0.0026
Epoch  9/12 | average loss 0.0012
Epoch 10/12 | average loss 0.0007
Epoch 11/12 | average loss 0.0006
Epoch 12/12 | average loss 0.0005


### 5. Evaluate the model on test data

We pick the class with `argmax`, just like in MNIST. Both labels appear equally often
in the test set, so always choosing the same option would give 50% accuracy.

In [21]:
def predict(input_tokens):
    with torch.no_grad():
        logits = forward(input_tokens)
        return logits.argmax(dim=-1)


def evaluate(loader):
    model.eval()
    correct_predictions = 0
    total_predictions = 0

    for batch_tokens, batch_labels in loader:
        predicted_labels = predict(batch_tokens.to(device))
        correct_predictions += (predicted_labels == batch_labels.to(device)).sum().item()
        total_predictions += len(batch_labels)

    return correct_predictions / total_predictions


test_accuracy = evaluate(test_loader)
print(f"Accuracy on test pairs: {test_accuracy:.1%}")

Accuracy on test pairs: 100.0%


#### See how swapping the order changes the answer

Below, the model gets all four variants of one test pair.
In the first and third text the answer options are in the same order,
so changing the finishing order should change both the class and the chosen name.

In [22]:
a, b = test_pairs[0]
answer_orders = [(a, b), (b, a), (a, b), (b, a)]

for (text, label), options in zip(make_examples([(a, b)]), answer_orders):
    input_tokens = torch.tensor([encode(text)], device=device)
    predicted_label = predict(input_tokens).item()

    print(text)
    print(f"Model → {options[predicted_label]} ({predicted_label})"
          f" | Correct answer → {options[label]} ({label})")

At the finish: Gniewomir, Andrzej. Who was first? Gniewomir or Andrzej?
Model → Gniewomir (0) | Correct answer → Gniewomir (0)
At the finish: Gniewomir, Andrzej. Who was first? Andrzej or Gniewomir?
Model → Gniewomir (1) | Correct answer → Gniewomir (1)
At the finish: Andrzej, Gniewomir. Who was first? Gniewomir or Andrzej?
Model → Andrzej (1) | Correct answer → Andrzej (1)
At the finish: Andrzej, Gniewomir. Who was first? Andrzej or Gniewomir?
Model → Andrzej (0) | Correct answer → Andrzej (0)


#### Live demo · turn off position information

For a moment we zero out the learned position embeddings of the trained network. Variants
that differ only in the finishing order then get the same representation at the position
of the first option. Check how the accuracy changes.

In [23]:
position_weights = model[1].embedding.weight

with torch.no_grad():
    saved_weights = position_weights.clone()
    try:
        position_weights.zero_()
        accuracy_without_positions = evaluate(test_loader)
    finally:
        position_weights.copy_(saved_weights)

print(f"With position embeddings:    {test_accuracy:.1%}")
print(f"Without position embeddings: {accuracy_without_positions:.1%}")

With position embeddings:    100.0%
Without position embeddings: 50.0%
